# 让 Agent 玩扫雷

本 Notebook 使用一个 4×4 扫雷棋盘作为 Agent 的运行环境。扫雷的每一步需要：观察棋盘状态、判断风险、选择动作（reveal / flag / unflag）、执行、读取反馈。这对应 Agent 的核心循环：

```text
Goal → Observe → Think → Use Tools → Read Feedback → Repeat
```

下面将按以下顺序展开：环境准备 → 模型启动 → Agent 概念拆解 → 逐步组装完整循环 → 运行观察。

## 1. Workshop

完整流程约 90 分钟：

| 阶段 | 核心内容 | 可观察现象 |
|---|---|---|
| 区别 | Chatbot 生成文本，Agent 执行动作并读取反馈 | 同一个任务，三种处理方式的输出差异 |
| 环境 | Jupyter Notebook 统一运行和观察 | 代码、解释和输出集中在同一页面 |
| 模型 | Gemma 4 E4B 通过 llama.cpp 提供 OpenAI-compatible API | 本地 endpoint 可被任意 HTTP 客户端调用 |
| 游戏 | 扫雷棋盘作为 Agent 的外部环境 | 棋盘状态因工具调用而变化 |
| Agent | toy-cli 风格循环：观察 → 候选 → 模型选择 → 工具执行 → 反馈 | 每轮输出一次 JSON 工具调用 |
| 复盘 | 小模型 Agent 的稳定性设计 | 候选动作、低温度、JSON 约束、fallback |


## 2. 运行环境

默认配置：

- Python 3.12
- AMD ROCm 环境
- `llama-server` endpoint：`http://127.0.0.1:8080/v1/chat/completions`
- 模型：Gemma 4 E4B-it Q4_K_M GGUF（约 5.4 GB）

## 3. 下载并启动 llama.cpp（ROCm 预编译版）

本实验使用 [lemonade-sdk/llamacpp-rocm](https://github.com/lemonade-sdk/llamacpp-rocm) 提供的预编译 ROCm 版本。

### 3.1 下载 llama.cpp

默认使用 **gfx1151**（Ryzen AI Max+ 395 / Radeon 8060S）：

```bash
# gfx1151 — Ryzen AI Max+ 395 / Radeon 8060S -> AMD AUP learning cloud
wget https://github.com/lemonade-sdk/llamacpp-rocm/releases/download/b1292/llama-b1292-ubuntu-rocm-gfx1151-x64.zip

# 如果使用 AMD Radeon Cloud 的 Radeon PRO W7900 或其他 gfx110X 显卡：
# wget https://github.com/lemonade-sdk/llamacpp-rocm/releases/download/b1292/llama-b1292-ubuntu-rocm-gfx110X-x64.zip
```

### 3.2 解压

```bash
unzip llama-b1292-ubuntu-rocm-gfx1151-x64.zip -d llama-server-rocm
chmod +x llama-server-rocm/llama-server
```

### 3.3 下载 Gemma 4 E4B GGUF 模型

**路线 A：ModelScope（国内直连，免登录）**

```bash
mkdir -p ./models
wget https://www.modelscope.cn/models/bartowski/google_gemma-4-E4B-it-GGUF/resolve/master/google_gemma-4-E4B-it-Q4_K_M.gguf -P ./models
```

**路线 B：Hugging Face**

```bash
mkdir -p ./models
wget https://huggingface.co/bartowski/google_gemma-4-E4B-it-GGUF/resolve/main/google_gemma-4-E4B-it-Q4_K_M.gguf -P ./models
```

### 3.4 启动 llama-server

```bash
# 使用llamacpp加载模型，--reasoning off 关闭思考模式
./llama-server-rocm/llama-server \
  -m ./models/google_gemma-4-E4B-it-Q4_K_M.gguf \
  -ngl 99 -c 65536 --reasoning off
```

启动成功后 API 地址为 `http://127.0.0.1:8080/v1/chat/completions`。

![llamacpp](../../../docs/public/images/05-amd-yes/minesweeper-agent/minesweeper_agent_1.png)

## 4. 验证 llama-server 连接

通过 `GET /v1/models` 确认 endpoint 可用。

In [1]:
%pip install -q requests

import requests

ENDPOINT = "http://127.0.0.1:8080"

try:
    resp = requests.get(f"{ENDPOINT}/v1/models", timeout=5)
    resp.raise_for_status()
    models = [m["id"] for m in resp.json().get("data", [])]
    print(f"llama-server 已连接，可用模型: {models}")
except Exception as e:
    print(f"连接失败: {e}")
    print("请先按照上一节说明启动 llama-server，再重新运行本 cell。")


Note: you may need to restart the kernel to use updated packages.
llama-server 已连接，可用模型: ['google_gemma-4-E4B-it-Q4_K_M.gguf']


## 5. Chatbot 与 Agent 的区别

Agent 的核心特征：围绕目标，通过工具与环境交互，并根据反馈修正行为。

| 对比项 | Chatbot | Agent |
|---|---|---|
| 输入 | 一段文本 | 目标 + 当前环境状态 |
| 输出 | 一段文本 | 一个可执行的工具调用 |
| 反馈来源 | 用户追问 | 工具返回值 / 环境状态变化 |
| 典型风险 | 回答不准确 | 调错工具、修改错误状态、死循环 |
| 稳定性手段 | 提示词优化 | 工具边界、候选动作、最大步数、fallback |

Agent 需要回答三个问题才能行动：

1. **当前状态是什么？** → Observation
2. **目标是什么？** → Goal
3. **可用动作有哪些？** → Tools

## 6. 扫雷环境

导入 4×4 扫雷游戏。Agent 可用的工具：

- `reveal(row, col)`：揭开一个格子，返回数字或触雷。
- `flag(row, col)`：标记一个格子为疑似地雷。
- `unflag(row, col)`：取消标记。

任务：反复观察棋盘并选择工具，直到所有非雷格子被揭开（胜利）或触雷（失败）。

In [2]:
from minesweeper_game import (
    MinesweeperGame,
    apply_decision,
    observe_for_llm,
    propose_minesweeper_actions,
    render_ascii,
)

game = MinesweeperGame(board_size=4, mine_count=3, seed=7)
print(render_ascii(game))


      1   2   3   4
  A   _   _   _   _
  B   _   _   _   _
  C   _   _   _   _
  D   _   _   _   _


运行上面的代码后，输出一个 4×4 棋盘：

```
      1   2   3   4
  A   _   _   _   _
  B   _   _   _   _
  C   _   _   _   _
  D   _   _   _   _
```

- `_` 表示未揭开的格子。
- 行标用字母（A-D），列标用数字（1-4）。
- 棋盘下藏着 3 颗地雷，位置由 `seed=7` 确定（固定种子保证可复现）。

此时 Agent 面对的是一个完全未知的环境——没有任何已揭开的数字可供推理，第一步只能选择一个位置尝试。

## 7. 手动执行一步

在 Agent 接管之前，先手动执行一次 `reveal`，观察棋盘状态变化和工具返回的反馈结构。

In [3]:
game, result = apply_decision(game, {"action": "reveal", "args": {"row": "A", "col": 1}})
print("工具反馈:", result)
print(render_ascii(game))


工具反馈: {'success': True, 'message': 'revealed 8 cell(s) from A1', 'outcome': 'playing'}
      1   2   3   4
  A   .   2   _   _
  B   .   3   _   _
  C   .   2   _   _
  D   .   1   _   _


执行 `reveal` 后，工具返回一个结构化反馈字典：

```python
{
  "success": True,           # 工具调用是否成功
  "message": "revealed 5 cell(s) from A1",  # 发生了什么
  "outcome": "playing"       # 游戏状态：playing / won / lost
}
```

棋盘变化：

- 数字格（如 `1`、`2`）表示该格子周围 8 个邻居中有多少颗雷。
- `.` 表示周围 0 颗雷，触发 flood-fill 自动展开相邻的零值格。
- `_` 仍然是未揭开的格子。

**知识点：Tool Result 的设计**

Agent 系统中，每个工具调用都应返回结构化反馈（而非纯文本），使 Agent 能够程序化地判断：

1. 动作是否成功执行。
2. 环境发生了什么变化。
3. 整体任务是否结束（won/lost/playing）。

## 8. Observation：Agent 看到什么

Agent 通过 `observe_for_llm(game)` 获取结构化观察窗口，包含：棋盘 ASCII、当前分数、允许的工具列表、上一次动作的反馈。

这比直接暴露 Python 对象更紧凑，也更适合作为 LLM 的输入。

In [4]:
import json

observation = observe_for_llm(game)
print(json.dumps(observation, ensure_ascii=False, indent=2))

{
  "board": "      1   2   3   4\n  A   .   2   _   _\n  B   .   3   _   _\n  C   .   2   _   _\n  D   .   1   _   _",
  "state": {
    "score": 80,
    "revealed_safe": 8,
    "safe_total": 13,
    "correct_flags": 0,
    "wrong_flags": 0,
    "turn": 1,
    "status": "playing"
  },
  "allowed_actions": [
    "reveal",
    "flag",
    "unflag"
  ],
  "last_action": {
    "turn": 1,
    "action": "reveal",
    "cell": "A1",
    "result": {
      "success": true,
      "message": "revealed 8 cell(s) from A1",
      "outcome": "playing"
    }
  }
}



`observe_for_llm(game)` 返回的字典包含四个字段：

| 字段 | 作用 |
|---|---|
| `board` | 当前棋盘的 ASCII 表示，作为模型的视觉输入 |
| `state` | 包含 score、revealed_safe、safe_total、turn、status 等进度信息 |
| `allowed_actions` | Agent 当前可以使用的工具列表 |
| `last_action` | 上一步的动作和结果，供 Agent 参考 |

**知识点：Observation 设计原则**

- 信息应足够 Agent 做出决策，但不应包含冗余数据。
- 结构化格式（JSON）比自然语言描述更适合作为 LLM 的输入——模型更容易从固定 key 中提取信息。
- `allowed_actions` 字段让 Agent 知道边界在哪里，避免尝试不存在的工具。

## 9. Candidate Actions：约束模型输出

小模型容易输出不存在的坐标、多个动作或非 JSON 格式。

解决方式：先用确定性 Python 逻辑（经典扫雷推理规则）生成一组合法候选动作，再让模型从中选择。这将开放式生成问题转化为选择问题。

In [5]:
candidates = propose_minesweeper_actions(game)
print(json.dumps(candidates, ensure_ascii=False, indent=2))

[
  {
    "action": "flag",
    "args": {
      "row": "A",
      "col": 3
    },
    "score": 0.95,
    "reason": "A2 shows 2; all hidden neighbors are mines."
  },
  {
    "action": "flag",
    "args": {
      "row": "B",
      "col": 3
    },
    "score": 0.95,
    "reason": "A2 shows 2; all hidden neighbors are mines."
  },
  {
    "action": "flag",
    "args": {
      "row": "C",
      "col": 3
    },
    "score": 0.95,
    "reason": "B2 shows 3; all hidden neighbors are mines."
  }
]



每个候选动作的结构：

| 字段 | 作用 |
|---|---|
| `action` | 工具名称：reveal / flag / unflag |
| `args` | 工具参数（坐标） |
| `score` | 确定性评分（0-1），越高越安全 |
| `reason` | 推理依据（来自经典扫雷逻辑） |

**知识点：候选动作的作用**

小模型（4B-8B）在自由生成时容易失败：编造不存在的坐标、输出多个动作、返回自然语言而非 JSON。

候选动作机制将问题从“生成”转化为“选择”：模型只需从 3-6 个合法选项中挑一个，大幅降低格式错误率。

`score` 字段的来源是确定性扫雷推理规则：

- 0.95：通过数字约束确定是雷 → flag。
- 0.90：通过数字约束确定安全 → reveal。
- 0.50：无确定推理，选择角落/边缘作为统计上更安全的猜测。

## 9.5 对比实验：让模型自己生成下一步

上一步用 Python 生成了候选动作。这里做一个对比：不提供候选列表，直接把棋盘状态发给模型，让它自由生成下一步动作。

观察模型输出：格式是否符合预期？坐标是否在棋盘范围内？是否只返回了一个动作？

In [6]:
import requests as _req
import re as _re

_endpoint = "http://127.0.0.1:8080/v1/chat/completions"

_SYSTEM_FOR_CONTRAST = """You are a Minesweeper player on a 4x4 board (rows A-D, columns 1-4).
Rules: number K = K adjacent mines. "." = 0 mines. "_" = unrevealed. "F" = flagged.
Analyze the board, then output ONE JSON: {"action": "reveal|flag|unflag", "args": {"row": "A", "col": 1}, "reason": "why"}"""

obs = observe_for_llm(game)
_user_prompt = (
    f"Board:\n{obs['board']}\n\n"
    f"Turn: {obs['state']['turn']}, Status: {obs['state']['status']}\n"
    f"Decide your next move."
)

print("=== user prompt ===")
print(_user_prompt)
print()

resp = _req.post(_endpoint, json={
    "model": "gemma-4-E4B-it-Q4_K_M",
    "messages": [
        {"role": "system", "content": _SYSTEM_FOR_CONTRAST},
        {"role": "user", "content": _user_prompt},
    ],
    "temperature": 0.3,
    "max_tokens": 8192,
}, timeout=120)
resp.raise_for_status()
data = resp.json()

message = data["choices"][0]["message"]
content = message.get("content", "")
reasoning = message.get("reasoning_content", "")
finish_reason = data["choices"][0].get("finish_reason", "")

print("=== model reasoning (thinking process) ===")
if reasoning:
    print(reasoning[:1500])
    if len(reasoning) > 1500:
        print(f"... (truncated, total {len(reasoning)} chars)")
else:
    print("(no reasoning_content)")

print(f"\n=== model final output (content) ===")
print(content if content.strip() else "(empty)")
print(f"\nfinish_reason: {finish_reason}")

# 尝试从 content 解析 JSON
model_decision = None
if content.strip():
    try:
        fenced = _re.search(r"```json\s*(.*?)\s*```", content, _re.S)
        payload = fenced.group(1) if fenced else content
        start = payload.find("{")
        end = payload.rfind("}")
        if start != -1 and end != -1:
            model_decision = json.loads(payload[start:end+1])
            print(f"\n=== parsed decision ===")
            print(json.dumps(model_decision, ensure_ascii=False, indent=2))
    except Exception as e:
        print(f"\nJSON parse failed: {e}")

# 对比 Section 9 的 Python 候选
print(f"\n{'='*50}")
print("=== comparison: model plan vs Python candidates ===")
print(f"\nPython top-3 candidates (from Section 9):")
for c in candidates[:3]:
    print(f"  {c['action']} {c['args']} (score={c['score']}) - {c['reason']}")

if model_decision:
    print(f"\nModel chose: {model_decision.get('action')} {model_decision.get('args')}")
    print(f"Model reason: {model_decision.get('reason', '(none)')}")
elif finish_reason == "length":
    print(f"\nModel used all tokens on reasoning ({len(reasoning)} chars) without producing final JSON.")
    print("This shows: small model spends too many tokens thinking, leaving no room for the answer.")
else:
    print(f"\nModel did not produce a valid action.")


=== user prompt ===
Board:
      1   2   3   4
  A   .   2   _   _
  B   .   3   _   _
  C   .   2   _   _
  D   .   1   _   _

Turn: 1, Status: playing
Decide your next move.

=== model reasoning (thinking process) ===
(no reasoning_content)

=== model final output (content) ===
The current board state is:
      1   2   3   4
  A   .   2   _   _
  B   .   3   _   _
  C   .   2   _   _
  D   .   1   _   _

We have several unrevealed cells (`_`) and some revealed cells with numbers. The cells in column 1 are all revealed as `.` (meaning 0 adjacent mines).

The most promising area to start is in the unrevealed cells, particularly around the known numbers, as revealing them might give us more information.

Let's look at cell A3. It is adjacent to A2 (2), B2 (3), and C2 (2).
*   A3 is adjacent to: A2(2), B3(_), A4(_), B2(3).
*   B3 is adjacent to: A3(_), B2(3), C3(_), B4(_).
*   C3 is adjacent to: B3(_), C2(2), D3(_), C4(_).
*   D3 is adjacent to: C3(_), D2(1), D4(_).

Since column 1 is al



将上面的模型输出与 Section 9 的 Python 候选对比：

| 对比项 | Section 9：Python 候选动作 | Section 9.5：模型自主生成 |
|---|---|---|
| 生成方式 | 确定性规则（数字约束推理） | 模型理解 prompt 后自由生成 |
| 输出数量 | 一次输出多个候选（排序 + 评分） | 一次只输出一个动作 |
| 坐标合法性 | 保证合法（仅遍历未操作的格子） | 依赖模型遵守 prompt 约束 |
| 推理依据 | 精确的扫雷规则（score 0.95/0.90/0.50） | 模型的 reason 字段（可能正确也可能模糊） |
| 计算成本 | 毫秒级，不消耗 token | 需要一次 LLM 调用 |
| 失败模式 | 不会失败 | 可能输出非 JSON / 坐标超范围 / 重复已揭开的格子 |

**核心区别：**

- Python 候选是 **确定性逻辑**：只要棋盘状态给定，输出永远正确、格式永远合法。
- 模型自主生成是 **概率性输出**：结果取决于 prompt 质量、模型能力和采样参数，可能正确也可能出错。

当模型输出正确时，它的 `reason` 字段可以提供自然语言解释；当输出错误时，整个 Agent 循环可能被中断或产生无效动作。Section 13 的做法是将两者结合：Python 生成候选保证合法性，模型从候选中选择并给出理由。

## 10. LLM Client：连接本地模型

`LlamaCppClient` 封装对 `llama-server` 的 HTTP 调用，对应 toy-cli 中 `LocalLLM` 的核心逻辑。

关键参数：

- `temperature=0.3`：降低输出随机性，减少格式错误。
- `max_tokens=8192`：单次工具调用最大输出token数量。
- `extract_json`：从模型回复中提取 JSON（模型有时会包裹 Markdown 代码块）。

In [7]:
import re
from dataclasses import dataclass


@dataclass
class LlamaCppClient:
    endpoint: str = "http://127.0.0.1:8080/v1/chat/completions"
    model: str = "gemma-4-E4B-it-Q4_K_M"
    temperature: float = 0.3

    def chat(self, system_prompt: str, user_prompt: str, max_tokens: int = 8192) -> str:
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": self.temperature,
            "max_tokens": max_tokens,
        }
        response = requests.post(self.endpoint, json=payload, timeout=120)
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]


def extract_json(text: str) -> dict:
    """从模型回复中提取 JSON 对象，兼容 ```json ... ``` 包裹格式。"""
    fenced = re.search(r"```json\s*(.*?)\s*```", text, re.S)
    payload = fenced.group(1) if fenced else text
    start = payload.find("{")
    end = payload.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in model output: {text[:200]}")
    return json.loads(payload[start : end + 1])


client = LlamaCppClient()


## 11. System Prompt：限制行动格式

System Prompt 约束模型的输出格式，而不是提升对话能力：

1. 只能从候选动作中选择一个。
2. 输出必须是指定 JSON 结构。
3. 不得编造棋盘外坐标。

分工：**模型负责选择，工具负责执行，程序负责边界验证。**

In [8]:
SYSTEM_PROMPT = """You are a careful Minesweeper Agent.
Choose exactly one action from the candidate list.
Return only JSON in this shape:
{"action": "reveal|flag|unflag", "args": {"row": "A", "col": 1}, "reason": "short reason"}
Do not invent coordinates outside the board."""


def build_user_prompt(game: MinesweeperGame) -> str:
    """将当前观察和候选动作组装为发送给模型的 user prompt。"""
    return json.dumps(
        {
            "goal": "Clear every non-mine cell without hitting a mine.",
            "observation": observe_for_llm(game),
            "candidates": propose_minesweeper_actions(game),
        },
        ensure_ascii=False,
        indent=2,
    )


### 知识补充：User Prompt 的组装结构

`build_user_prompt(game)` 将以下三部分组装为 JSON 发送给模型：

```json
{
  "goal": "Clear every non-mine cell...",
  "observation": { ... },
  "candidates": [ ... ]
}
```

对应 Agent 决策的三个输入：

1. **Goal**：任务目标，保持不变。
2. **Observation**：每轮更新的环境状态。
3. **Candidates**：每轮更新的合法动作池。

模型的任务是：阅读当前状态，从候选中选择一个最合理的动作，输出指定 JSON 格式。

## 12. 单步决策：模型输出 → 工具调用

`choose_move` 的流程：

1. 生成候选动作列表。
2. 将观察和候选发送给模型。
3. 解析模型返回的 JSON。
4. 解析失败时，fallback 到得分最高的候选动作。

Fallback 机制确保单次 LLM 输出异常不会中断整个循环。

In [9]:
def choose_move(game: MinesweeperGame, llm_client: LlamaCppClient) -> dict:
    """一次 Agent 决策：生成候选 → 模型选择 → 解析 JSON → fallback。"""
    candidates = propose_minesweeper_actions(game)
    if not candidates:
        return {"action": "stop", "args": {}, "reason": "no candidate moves"}

    try:
        raw = llm_client.chat(SYSTEM_PROMPT, build_user_prompt(game))
        decision = extract_json(raw)
    except Exception as exc:
        # fallback: 使用确定性逻辑推荐的最优候选
        best = candidates[0]
        decision = {
            "action": best["action"],
            "args": best["args"],
            "reason": f"fallback to top candidate (LLM error: {exc})",
        }

    return decision


## 13. 完整 Agent 循环

将上述组件组装为完整循环。每一步输出：

- **Observe**：当前棋盘状态。
- **Plan**：候选动作列表（Python 生成）。
- **Act**：模型选择的工具调用。
- **Feedback**：工具执行结果。

最多运行 10 步。

In [10]:
agent_game = MinesweeperGame(board_size=4, mine_count=3, seed=11)

for step in range(1, 11):
    if agent_game.status != "playing":
        break

    print(f"\n{'='*40}")
    print(f"Step {step}: Observe")
    print(render_ascii(agent_game))

    print("\nPlan: candidate actions")
    print(json.dumps(propose_minesweeper_actions(agent_game), ensure_ascii=False, indent=2))

    decision = choose_move(agent_game, client)
    print(f"\nAct: {decision}")

    if decision.get("action") == "stop":
        break

    agent_game, result = apply_decision(agent_game, decision)
    print(f"Feedback: {result}")

print(f"\n{'='*40}")
print("Final board:")
print(render_ascii(agent_game, reveal_mines=True))
print(f"Result: {agent_game.status}, Score: {observe_for_llm(agent_game)['state']}")



Step 1: Observe
      1   2   3   4
  A   _   _   _   _
  B   _   _   _   _
  C   _   _   _   _
  D   _   _   _   _

Plan: candidate actions
[
  {
    "action": "reveal",
    "args": {
      "row": "A",
      "col": 1
    },
    "score": 0.5,
    "reason": "No certain move yet; choose a corner/edge as a simple fallback."
  },
  {
    "action": "reveal",
    "args": {
      "row": "A",
      "col": 4
    },
    "score": 0.5,
    "reason": "No certain move yet; choose a corner/edge as a simple fallback."
  },
  {
    "action": "reveal",
    "args": {
      "row": "D",
      "col": 1
    },
    "score": 0.5,
    "reason": "No certain move yet; choose a corner/edge as a simple fallback."
  }
]

Act: {'action': 'reveal', 'args': {'row': 'A', 'col': 1}, 'reason': 'Starting with a corner reveal as there is no initial information to deduce a safe move.'}
Feedback: {'success': True, 'message': 'revealed 11 cell(s) from A1', 'outcome': 'playing'}

Step 2: Observe
      1   2   3   4
  A   .   .



完整循环的每一步输出四个部分，对应 Agent 循环的四个阶段：

```
========================================
Step N: Observe          ← 当前棋盘 ASCII

Plan: candidate actions  ← Python 生成的候选列表

Act: {...}               ← 模型选择的工具调用 JSON
Feedback: {...}          ← 工具执行后的环境反馈
========================================
```

观察要点：

- 每步的 `candidates` 会根据当前棋盘状态动态变化——已揭开的数字格提供新的推理约束。
- `Act` 中的 `reason` 字段是模型给出的决策理由。
- 如果 `Act` 中出现 `fallback to top candidate`，说明模型输出了无法解析的格式，程序自动使用了最安全的候选。
- 游戏结束时（won 或 lost），最终棋盘显示所有地雷位置（`*`）。

**知识点：Agent Loop 的工程特征**

- 循环有明确的终止条件（游戏结束或达到最大步数）。
- 每轮只执行一个动作，执行后立即读取反馈——这是“单步 Agent”的标准模式。
- 状态完全由环境维护（`MinesweeperGame` 对象），Agent 不保存内部记忆。
- 候选动作每轮重新生成，反映最新棋盘状态。

## 13.5 完全自主的 Agent 循环

Section 13 中，Agent 的每一步都从 Python 生成的候选列表中选择。这里去掉候选列表，让模型完全自主地完成整局游戏。实现上包含三个关键设计：

1. **增强提示词**：System Prompt 包含扫雷规则、推理策略、坐标范围、输出格式约束。
2. **动作历史**：每轮将已执行的所有动作（包含成功和失败）传入 prompt，防止模型重复选择同一个格子。
3. **重试机制**：JSON 解析失败或坐标超范围时，将错误信息反馈给模型并重新请求（最多 2 次），只执行合法的动作。


In [ ]:
FREE_AGENT_SYSTEM = """You are an expert Minesweeper Agent. You play on a 4x4 board with 3 hidden mines.

## Game Rules
- The board has rows A-D and columns 1-4. Valid coordinates: A1 to D4.
- A revealed number K means exactly K of its 8 adjacent cells contain mines.
- "." is a revealed cell , means 0 adjacent mines (all neighbors are safe).
- "_" is an unrevealed cell. 
- "F" is a flagged cell.
- You WIN by revealing all 13 non-mine cells. You LOSE if you reveal a mine.

## Strategy
1. If a number K has exactly K unrevealed+unflagged neighbors, those are ALL mines -> flag them.
2. If a number K already has K flagged neighbors, remaining hidden neighbors are SAFE -> reveal them.
3. Otherwise guess a corner or edge cell that has NOT been tried before.
4. NEVER repeat an action from the history. Choose a DIFFERENT cell each turn.

## Output Format
Return EXACTLY one JSON object, no other text:
{"action": "reveal|flag|unflag", "args": {"row": "A", "col": 1}, "reason": "brief"}

## Constraints
- row: A, B, C, D | col: 1, 2, 3, 4
- Do NOT choose any cell that appears in the action history below."""


def free_agent_prompt(game: MinesweeperGame, action_log: list, error_feedback: str = None) -> str:
    obs = observe_for_llm(game)
    prompt = (
        f"Board (turn {obs['state']['turn']}):\n"
        f"{obs['board']}\n\n"
        f"Progress: {obs['state']['revealed_safe']}/{obs['state']['safe_total']} safe cells revealed\n"
    )

    if action_log:
        prompt += f"\n## Action History (do NOT repeat any of these cells)\n"
        for entry in action_log[-10:]:
            prompt += f"- {entry}\n"
        prompt += "\n"

    if error_feedback:
        prompt += (
            f"## PREVIOUS ATTEMPT FAILED\n"
            f"{error_feedback}\n"
            f"Choose a DIFFERENT cell.\n\n"
        )

    prompt += "Pick ONE new action (not in history)."
    return prompt


def free_choose_move_with_retry(game: MinesweeperGame, llm_client: LlamaCppClient, action_log: list, max_retries: int = 2) -> dict:
    from minesweeper_game import parse_coord

    error_feedback = None

    for attempt in range(1 + max_retries):
        try:
            raw = llm_client.chat(FREE_AGENT_SYSTEM, free_agent_prompt(game, action_log, error_feedback))
            decision = extract_json(raw)

            action = decision.get("action")
            if action not in ("reveal", "flag", "unflag"):
                raise ValueError(f"invalid action '{action}'")

            args = decision.get("args", {})
            parse_coord(args, game.board_size)

            if attempt > 0:
                print(f"  (retry {attempt} succeeded)")
            return decision

        except Exception as exc:
            error_feedback = str(exc)
            if attempt < max_retries:
                print(f"  (attempt {attempt+1} failed: {exc} -> retrying)")
            else:
                return {"action": "stop", "args": {}, "reason": f"retries exhausted: {exc}"}


# --- Run ---
free_game = MinesweeperGame(board_size=4, mine_count=3, seed=11)
action_log = []  # 维护完整动作记录（包含成功和失败）
invalid_moves = 0

for step in range(1, 20):
    if free_game.status != "playing":
        break

    print(f"\n{'='*50}")
    print(f"Step {step}")
    print(render_ascii(free_game))

    decision = free_choose_move_with_retry(free_game, client, action_log, max_retries=2)
    print(f"Action: {json.dumps(decision, ensure_ascii=False)}")

    if decision.get("action") == "stop":
        print("(Agent stopped)")
        break

    # 记录动作（无论成功还是失败）
    cell_str = f"{decision['args'].get('row','?')}{decision['args'].get('col','?')}"
    free_game, result = apply_decision(free_game, decision)

    if result.get("success"):
        action_log.append(f"{decision['action']} {cell_str} -> {result['message']}")
        print(f"  -> {result['message']}")
    else:
        invalid_moves += 1
        action_log.append(f"{decision['action']} {cell_str} -> FAILED: {result['message']}")
        print(f"  [invalid] {result['message']}")

print(f"\n{'='*50}")
print("Final board:")
print(render_ascii(free_game, reveal_mines=True))
print(f"\nResult: {free_game.status}")
print(f"Turns played: {free_game.turn}")
print(f"Invalid moves: {invalid_moves}")
final_obs = observe_for_llm(free_game)
print(f"Safe cells revealed: {final_obs['state']['revealed_safe']}/{final_obs['state']['safe_total']}")



Step 1
      1   2   3   4
  A   _   _   _   _
  B   _   _   _   _
  C   _   _   _   _
  D   _   _   _   _
Action: {"action": "reveal", "args": {"row": "A", "col": 1}, "reason": "Starting move: Reveal an unrevealed cell to gain initial information."}
  -> revealed 11 cell(s) from A1

Step 2
      1   2   3   4
  A   .   .   .   .
  B   1   1   .   .
  C   _   3   2   1
  D   _   _   _   _
Action: {"action": "reveal", "args": {"row": "A", "col": 2}, "reason": "Start revealing unknown cells to gain more information."}
  [invalid] A2 already revealed

Step 3
      1   2   3   4
  A   .   .   .   .
  B   1   1   .   .
  C   _   3   2   1
  D   _   _   _   _
Action: {"action": "reveal", "args": {"row": "B", "col": 3}, "reason": "B3 is an unrevealed cell adjacent to several known numbers (B2=1, C2=3, C3=2, C4=1), helping to constrain mine locations."}
  [invalid] B3 already revealed

Step 4
      1   2   3   4
  A   .   .   .   .
  B   1   1   .   .
  C   _   3   2   1
  D   _   _   _   _
A



| 对比项 | Section 13（候选动作） | Section 13.5（完全自主） |
|---|---|---|
| 动作来源 | Python 确定性规则生成候选 → 模型选择 | 模型自己分析棋盘并生成动作 |
| 无效动作 | 候选列表只包含可操作的格子，不会出现无效动作 | 可能选择已揭开的格子（如 A2），需要历史记录纠正 |
| 格式稳定性 | 模型复制候选的 JSON 结构，几乎不会出错 | 可能出现 JSON 解析失败（实际运行中 Step 5 触发了重试） |
| 推理质量 | 确定性规则：0.95 分的 flag 必定正确 | 模型推理可能正确（flag C1）也可能误判（reveal D2 触雷） |
| 容错机制 | fallback 到最优候选 | 重试 + 错误反馈 + 历史记录防止重复 |
| 实际结果 | 稳定推进，很少无效动作 | 4 步有效推进，1 次无效动作，1 次 JSON 重试，最终触雷 |

**关键观察：**

- Step 2 模型选了已揭开的 A2（无效），但因为失败记录被写入 action_log，Step 3 开始模型不再重复该格子。
- Step 3 模型正确地 flag 了 C1（最终确认是雷），说明在简单场景下模型具备基本扫雷推理能力。
- Step 5 模型选择了 D2（实际是雷）——这是扫雷的随机性，在没有确定性推理依据时，模型和人类一样只能猜测。

**结论：**

- 自主式 Agent 能够完成任务，但稳定性低于候选式，需要额外的工程护栏（历史记录、重试、错误反馈）。
- 候选式 Agent 永远不会选择已揭开的格子，因为候选列表本身就过滤了这些情况。
- 两种模式的核心循环（Observe → Think → Act → Feedback）相同，区别在于 Think 阶段的自由度和安全边界的位置。


## 14. 复盘

本实验展示的是 Agent 系统的基本结构：

| 组件 | 本实验中的实现 |
|---|---|
| Goal | 揭开所有非雷格子 |
| State | 4×4 棋盘 + 分数 + 历史动作 |
| Tools | reveal / flag / unflag |
| Policy | Gemma 4 E4B 从候选动作中选择 |
| Feedback | 工具返回值（成功 / 失败 / 胜利 / 触雷） |
| Guardrails | 候选动作约束、低温度、JSON 格式、最大步数、fallback |

## 15. 常见问题

| 现象 | 可能原因 | 处理方式 |
|---|---|---|
| 连接不上 `127.0.0.1:8080` | `llama-server` 没启动或端口不同 | 用 `curl http://127.0.0.1:8080/v1/models` 测试 |
| 模型输出不是 JSON | temperature 太高 / 模型不稳定 | 降低 temperature，保留 fallback |
| Agent 重复同一动作 | prompt 约束不足，或候选动作未更新 | 打印 candidates 检查 |
| Agent 触雷 | 扫雷有随机性，初期无确定安全格 | 重点观察循环结构，而非胜率 |
| 找不到 `minesweeper_game` 模块 | 工作目录不在 `src/amd-yes/toy-cli` | 切到该目录或在首 cell 加 `sys.path` |
